# Train a JATE ATE Tagger

Fine-tune a transformer model (XLM-R) for automatic term extraction using BIO sequence labelling on the [ACTER dataset](https://github.com/AylaRT/ACTER).

**What this notebook does:**
1. Installs dependencies
2. Downloads the ACTER dataset (4 domains, English)
3. Fine-tunes XLM-RoBERTa-base for token classification
4. Evaluates on held-out domain (heart failure)
5. Saves the model for use with JATE
6. (Optional) Uploads to HuggingFace Hub

**Time:** ~30-60 minutes on Colab T4 GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ziqizhang/jate/blob/dev/examples/train_bert_tagger.ipynb)

## 1. Setup

In [ ]:
# Install JATE from GitHub (feature branch) with neural extras
# NOTE: Change @feature/issue-92-bert-tagger to @dev after this branch is merged
# We use --no-deps for jate to avoid numpy version conflicts with Colab's environment,
# then install the neural extras separately.
!pip install -q --no-deps "jate @ git+https://github.com/ziqizhang/jate.git@feature/issue-92-bert-tagger"
!pip install -q transformers torch datasets seqeval spacy pandas

# Download spaCy model (needed by JATE's dataset loaders)
!python -m spacy download -q en_core_web_sm

# Verify GPU
import torch
device = 0 if torch.cuda.is_available() else -1
print(f"Device: {'GPU (' + torch.cuda.get_device_name(0) + ')' if device >= 0 else 'CPU'}")
if device == -1:
    print("WARNING: No GPU detected. Training will be very slow (~hours instead of ~30 min).")

print("Setup complete!")

## 2. Download ACTER dataset

In [ ]:
from jate.datasets.acter import Acter

# This downloads ~16 MB from GitHub on first run
ds = Acter()
print(f"ACTER loaded: {len(ds.documents)} documents, {len(ds.gold_terms)} gold terms")

## 3. Load IOB training data

ACTER provides token-level IOB annotations. We train on 3 domains (corruption, dressage, wind energy) and evaluate on the held-out domain (heart failure) — matching the TermEval 2020 shared task protocol.

In [ ]:
import sys
from pathlib import Path

# The training script's data loading functions
# We inline them here so the notebook is self-contained

LABEL_LIST = ["O", "B", "I"]
LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}
ID2LABEL = {i: label for i, label in enumerate(LABEL_LIST)}
ACTER_DOMAINS = ("corp", "equi", "htfl", "wind")


def load_iob_file(path):
    sentences = []
    tokens, labels = [], []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            if tokens:
                sentences.append({"tokens": tokens, "labels": labels})
                tokens, labels = [], []
            continue
        parts = line.split("\t")
        if len(parts) >= 2:
            tokens.append(parts[0])
            label = parts[1].strip()
            labels.append(label if label in LABEL2ID else "O")
        else:
            tokens.append(parts[0])
            labels.append("O")
    if tokens:
        sentences.append({"tokens": tokens, "labels": labels})
    return sentences


def load_acter_domain(domain, acter_dir):
    iob_dir = (
        acter_dir / "en" / domain / "annotated" / "annotations"
        / "sequential_annotations" / "iob_annotations" / "without_named_entities"
    )
    if not iob_dir.is_dir():
        print(f"WARNING: {iob_dir} not found")
        return []
    sentences = []
    for tsv in sorted(iob_dir.glob("*.tsv")):
        sentences.extend(load_iob_file(tsv))
    return sentences


# Load data
acter_dir = Path.home() / ".jate" / "datasets" / "acter"
eval_domain = "htfl"  # held-out for TermEval 2020 comparability

train_sents, eval_sents = [], []
for domain in ACTER_DOMAINS:
    sents = load_acter_domain(domain, acter_dir)
    print(f"  {domain}: {len(sents)} sentences")
    if domain == eval_domain:
        eval_sents.extend(sents)
    else:
        train_sents.extend(sents)

print(f"\nTrain: {len(train_sents)} sentences")
print(f"Eval:  {len(eval_sents)} sentences ({eval_domain})")

# Label distribution
from collections import Counter
all_labels = Counter()
for s in train_sents + eval_sents:
    all_labels.update(s["labels"])
print(f"Labels: {dict(all_labels)}")
term_pct = (all_labels['B'] + all_labels['I']) / sum(all_labels.values()) * 100
print(f"Term tokens: {term_pct:.1f}%")

## 4. Prepare HuggingFace datasets

In [ ]:
from datasets import Dataset as HFDataset

train_ds = HFDataset.from_list(train_sents)
eval_ds = HFDataset.from_list(eval_sents)

print(f"Train dataset: {train_ds}")
print(f"Eval dataset:  {eval_ds}")

## 5. Load model and tokeniser

Change `MODEL_NAME` below to use a different base model (e.g., `roberta-base` for English-only, or `dmis-lab/biobert-base-cased` for biomedical text).

In [ ]:
from transformers import AutoModelForTokenClassification, AutoTokenizer

MODEL_NAME = "xlm-roberta-base"  # Change this to try other models

print(f"Loading {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
print(f"Model parameters: {model.num_parameters():,}")

## 6. Tokenise and align labels

When a word is split into subword tokens, only the first subtoken gets the BIO label. The rest get `-100` (ignored in loss computation).

In [ ]:
def tokenize_and_align(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=512,
    )
    all_labels = []
    for i, labels in enumerate(examples["labels"]):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word_id:
                label_ids.append(LABEL2ID[labels[word_id]])
            else:
                label_ids.append(-100)  # subword continuation
            prev_word_id = word_id
        all_labels.append(label_ids)
    tokenized["labels"] = all_labels
    return tokenized


print("Tokenising ...")
train_tokenized = train_ds.map(tokenize_and_align, batched=True, remove_columns=train_ds.column_names)
eval_tokenized = eval_ds.map(tokenize_and_align, batched=True, remove_columns=eval_ds.column_names)
print(f"Train: {len(train_tokenized)} examples")
print(f"Eval:  {len(eval_tokenized)} examples")

## 7. Train

In [ ]:
import numpy as np
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from transformers import DataCollatorForTokenClassification, Trainer, TrainingArguments


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)
    true_labels, pred_labels = [], []
    for pred_seq, label_seq in zip(predictions, labels):
        true_sent, pred_sent = [], []
        for p, l in zip(pred_seq, label_seq):
            if l == -100:
                continue
            true_sent.append(ID2LABEL[l])
            pred_sent.append(ID2LABEL[p])
        true_labels.append(true_sent)
        pred_labels.append(pred_sent)
    return {
        "precision": precision_score(true_labels, pred_labels),
        "recall": recall_score(true_labels, pred_labels),
        "f1": f1_score(true_labels, pred_labels),
    }


# --- Hyperparameters (adjust as needed) ---
EPOCHS = 10
BATCH_SIZE = 16
LEARNING_RATE = 5e-5
OUTPUT_DIR = "./jate-ate-model"

training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

print(f"Training {MODEL_NAME} for {EPOCHS} epochs ...")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Output: {OUTPUT_DIR}")
trainer.train()

## 8. Evaluate

In [ ]:
# Final evaluation on held-out domain
metrics = trainer.evaluate()
print(f"\nResults on {eval_domain} (held-out):")
print(f"  Precision: {metrics['eval_precision']:.4f}")
print(f"  Recall:    {metrics['eval_recall']:.4f}")
print(f"  F1:        {metrics['eval_f1']:.4f}")

# Detailed classification report
predictions = trainer.predict(eval_tokenized)
preds = np.argmax(predictions.predictions, axis=2)
true_labels, pred_labels = [], []
for pred_seq, label_seq in zip(preds, predictions.label_ids):
    true_sent, pred_sent = [], []
    for p, l in zip(pred_seq, label_seq):
        if l == -100:
            continue
        true_sent.append(ID2LABEL[l])
        pred_sent.append(ID2LABEL[p])
    true_labels.append(true_sent)
    pred_labels.append(pred_sent)

print("\nDetailed report:")
print(classification_report(true_labels, pred_labels))

## 9. Save model

In [ ]:
save_dir = f"{OUTPUT_DIR}/model"
print(f"Saving to {save_dir} ...")
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print("Done!")

## 10. Test with JATE

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from jate.algorithms.bert_tagger import BertTagger

tagger = BertTagger(save_dir, device=device)

text = (
    "Corruption in public procurement is a major challenge for governments. "
    "Bribery and money laundering undermine the rule of law. "
    "Anti-corruption agencies work to prevent conflicts of interest."
)

result = tagger.tag(text)

print(f"Extracted {len(list(result))} terms:\n")
for term in result:
    spans_str = ", ".join(f'"{text[s.start:s.end]}"' for s in term.spans)
    print(f"  {term.string:30s}  score={term.score:.4f}  {spans_str}")

## 11. (Optional) Upload to HuggingFace Hub

Uncomment and run the cell below to upload your trained model. You'll need a HuggingFace account and access token.

In [ ]:
# # Login to HuggingFace (you'll be prompted for a token)
# from huggingface_hub import login
# login()

# # Upload
# HUB_REPO = "ziqizhang/jate-ate-xlmr"  # Change to your repo
# model.push_to_hub(HUB_REPO)
# tokenizer.push_to_hub(HUB_REPO)
# print(f"Uploaded to https://huggingface.co/{HUB_REPO}")